# Поверхностные координаты электродов ТТРКГ: анатомические ориентиры и геометрическая проверка

**Научный статус:** программный и геометрический прототип в рамках разработки
методики ТТРКГ. Интерфейс позволяет задать модельный монтаж, но выбранные в
браузере координаты не считаются фактическим положением электродов и не
являются физиологической или экспериментальной валидацией.

Ноутбук решает общую задачу размещения четырёх электродов ТТРКГ на наружной
поверхности тела. Расположение на руках является одним из возможных профилей,
а не ограничением метода.

Границы ответственности документов серии `40` разделены следующим образом:

1. `40.02` проверяет частный параметрический монтаж на искусственных
   продолжениях рук, четыре модели контакта и теорему взаимности.
2. `40.03` задаёт общий способ выбора анатомических ориентиров и координат на
   поверхности тела, формирует контракт монтажа и определяет проверки перед
   новым конечно-элементным расчётом (FEM).
3. `40.04` и `40.14` могут рассчитывать чувствительность только после принятия
   конкретного монтажа соответствующего эксперимента.


## Измерительная схема и область применения

Интерфейс предназначен только для четырёхэлектродной ТТРКГ экспериментов 2–3.
Центры выбираются в порядке `I_plus`, `V_plus`, `V_minus`, `I_minus`, где
`I_plus` и `I_minus` образуют токовую пару, а `V_plus` и `V_minus` —
измерительную пару.

Электроды могут быть расположены на любых требуемых участках наружной
поверхности тела. Интерфейс не задаёт боковые сборки мягких тканей, монтаж
РЕО32 эксперимента 1 или стандартную ТРКГ. Эти схемы имеют другие
экспериментальные задачи и требуют самостоятельных правил размещения.

Непосредственно измеренные физиологические сигналы в ноутбуке не используются.
Его входами служат наружная поверхность индивидуальной модели, справочные
анатомические слои и ручное решение пользователя о положении ориентиров или
центров электродов.


## Математическое определение поверхностных координат

Основной режим `explicit_points` сохраняет четыре независимо выбранные
декартовы координаты. Они служат предварительными центрами контактов и затем
повторно проецируются на наружную границу расчётной FEM-сетки.

Дополнительный режим `symmetric_paths` использует два независимо размеченных
поверхностных пути $\gamma_+$ и $\gamma_-$. Каждый путь начинается в своём
анатомическом ориентире и параметризуется накопленной длиной $s$ вдоль
поверхности. Координаты электродов определяются выражениями

$$
\begin{aligned}
\mathbf r_{V+}&=\gamma_+(d_{\mathrm{in}}), &
\mathbf r_{I+}&=\gamma_+(d_{\mathrm{in}}+d_{\mathrm{out}}),\\
\mathbf r_{V-}&=\gamma_-(d_{\mathrm{in}}), &
\mathbf r_{I-}&=\gamma_-(d_{\mathrm{in}}+d_{\mathrm{out}})
\end{aligned},
$$

где $\mathbf r$ — декартовы координаты центра электрода, мм;
$d_{\mathrm{in}}$ — расстояние от соответствующего анатомического ориентира
до внутреннего потенциального электрода, мм; $d_{\mathrm{out}}$ — расстояние
от внутреннего потенциального до внешнего токового электрода, мм; знаки $+$ и
$-$ обозначают стороны I+/V+ и I-/V-.

Симметрия означает равенство двух расстояний вдоль независимо выбранных
поверхностных путей. Координаты одной стороны не отражаются на другую, поэтому
метод не предполагает зеркальной симметрии анатомии.


## Причина перехода от абсолютных координат к поверхности

Исторический монтаж задавал центры абсолютными координатами около внешних
торцов и мест соединения искусственных рук с туловищем. На прежней сетке
токовые площадки захватывали грани разных участков поверхности и получались
почти в семь раз больше целевой площади. Сохранённый для этой постановки
передаточный импеданс 130,16 Ом не используется как базовый результат.

В `40.02` построена и технически принята новая 1-мм сетка для ограниченного
набора параметрических положений на руках. На ней выполнены 16 прямых и
взаимных сценариев, а 64 электродных патча прошли заданные геометрические
критерии. Эта проверка подтверждает работоспособность профиля монтажа на руках, но не
принимает автоматически любую точку, выбранную на поверхности тела.

При пробном переносе прежнего начала отсчёта в систему `surface_path_v1`
максимальное смещение центра составило 7,85 мм. Следовательно, положение
анатомического ориентира нельзя восстанавливать автоматически по старой
плоскости соединения. Ориентиры должны быть выбраны и зафиксированы автором.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

output_candidates = [
    Path("../MATLAB_TRKG4_real_subjects/output"),
    Path("MATLAB_TRKG4_real_subjects/output"),
]
OUTPUT = next((path.resolve() for path in output_candidates if path.exists()), None)
if OUTPUT is None:
    raise FileNotFoundError("Не найдена папка MATLAB_TRKG4_real_subjects/output")

acceptance = json.loads(
    (OUTPUT / "accepted_arm_parameter_mesh_acceptance.json").read_text(encoding="utf-8")
)
surface_check = json.loads(
    (OUTPUT / "nik_surface_coordinate_validation.json").read_text(encoding="utf-8")
)

status_table = pd.DataFrame(
    [
        ("Статус 1-мм сетки", "принята для разработки параметрического монтажа на руках"),
        ("Сценарии FEM в 40.02", acceptance["matlab"]["scenarios"]),
        ("Проверенные электродные патчи", acceptance["matlab"]["patches"]),
        (
            "Максимальная относительная ошибка взаимности",
            f'{acceptance["matlab"]["max_reciprocity_relative_error"]:.3e}',
        ),
        (
            "Максимальное смещение при переносе в surface_path_v1, мм",
            f'{surface_check["maximum_centre_difference_mm"]:.2f}',
        ),
        (
            "Повторный FEM-расчёт после переноса координат",
            "не выполнялся" if not surface_check["fem_forward_solve_performed"] else "выполнялся",
        ),
    ],
    columns=["Показатель", "Сохранённый результат"],
)
display(status_table)


,Показатель,Сохранённый результат
0,Статус 1-мм сетки,принята для разработки параметрического монтаж...
1,Сценарии FEM в 40.02,16
2,Проверенные электродные патчи,64
3,Максимальная относительная ошибка взаимности,5.921e-16
4,Максимальное смещение при переносе в surface_p...,7.85
5,Повторный FEM-расчёт после переноса координат,не выполнялся


## Анатомическая визуализация и выбор точек

Интерактивный интерфейс показывает в одной системе координат наружную
поверхность тела, скелет, лёгкие и сердце. Для добровольца Nix слой сердца
соответствует сердцу целиком, поскольку отдельная сегментация крови
отсутствует. Внутренние структуры служат только анатомическими ориентирами и
не участвуют в выборе поверхностных координат.

Поверхность тела по умолчанию непрозрачна. Видимость и непрозрачность каждого
анатомического слоя регулируются независимо. Условная светотень по нормалям
вершин облегчает восприятие рельефа, но не является картой чувствительности
или результатом FEM. Для браузерного отображения используется децимированная
копия поверхностей; расчётная сетка при этом не изменяется.

В свободном режиме пользователь последовательно выбирает четыре центра. В
симметричном режиме сначала размечаются два поверхностных пути, после чего
координаты вычисляются по общим значениям $d_{\mathrm{in}}$ и
$d_{\mathrm{out}}$. Кнопка симметричного размещения не ограничивает выбор
руками: два пути могут лежать на других участках наружной поверхности, если
такой монтаж соответствует исследуемой схеме ТТРКГ.


In [2]:
from pathlib import Path
from IPython.display import HTML, IFrame, display

PICKER_RELATIVE_URL = (
    "../MATLAB_TRKG4_real_subjects/output/"
    "nik_surface_landmark_picker.html"
)
picker_candidates = [
    Path(PICKER_RELATIVE_URL),
    Path("MATLAB_TRKG4_real_subjects/output/nik_surface_landmark_picker.html"),
]
picker_path = next((candidate.resolve() for candidate in picker_candidates if candidate.exists()), None)
if picker_path is None:
    raise FileNotFoundError(
        "Не найден nik_surface_landmark_picker.html. "
        "Сначала выполните tools/build_surface_landmark_picker.py."
    )

display(HTML(
    f'<p><a href="{PICKER_RELATIVE_URL}" target="_blank" rel="noopener">'
    'Открыть интерактивный монтаж ТТРКГ в отдельной вкладке</a>.</p>'
))
display(IFrame(src=PICKER_RELATIVE_URL, width="100%", height=1100))
display(HTML(
    '<p><em>Если просмотрщик блокирует интерактивные iframe, '
    'откройте инструмент по ссылке над визуализацией.</em></p>'
))


## Экспорт и передача координат в MATLAB

Интерфейс сохраняет файл `ttrkg_surface_montage.json` по контракту
`trkg4_ttrkg_surface_v3`. В контракт входят режим размещения, роли электродов,
выбранные точки или направляющие пути и метрические параметры. Абсолютные
локальные пути и метаданные КТ в файл не записываются.

Координаты из браузера имеют статус модельных кандидатов. Перед прямым
расчётом MATLAB должен повторно привязать их к полной расчётной поверхности,
построить контактные площадки и проверить:

1. расстояние исходной точки до поверхности и смещение центра площадки;
2. фактическую площадь каждого контакта;
3. отсутствие общих узлов, граней и перекрытия между площадками;
4. согласованность локальных нормалей и отсутствие перехода через резкое
   ребро поверхности;
5. выполнение теоремы взаимности после изменения монтажа.

Только после этих проверок допускается расчёт передаточного импеданса и карт
чувствительности для выбранного монтажа.


## Результат и нерешённые вопросы

Общий интерфейс размещения электродов на поверхности тела реализован. При
автоматизированной проверке первый выбор точки не приводил к зависанию, оба
режима формировали полный JSON-контракт, а ошибок JavaScript не обнаружено.
Этот результат подтверждает программную работоспособность интерфейса, но не
пригодность выбранного пользователем монтажа.

Принятая 1-мм сетка подтверждена только для испытанного диапазона положений на
искусственных продолжениях рук. Для монтажа на других участках поверхности
необходимо отдельно проверить локальную дискретизацию и возможность построить
контакты требуемой формы и площади.

Следующий содержательный шаг состоит в авторском выборе четырёх центров либо
двух пар соответствующих ориентиров и путей. После сохранения контракта следует
выполнить геометрический контроль в MATLAB, проверку взаимности и расчёт
передаточного импеданса. До этого момента `40.03` завершает программную часть
задания координат, но не завершает принятие физического монтажа ТТРКГ.
